# BERT Fine-tuning for Multiclass Sentiment Analysis

**Team:** Lost in Translation  
**Dataset:** Sp1786/multiclass-sentiment-analysis-dataset  
**Assignment:** Transfer Learning - Banana Assignment

---

## Overview
This notebook demonstrates fine-tuning a BERT model for multiclass sentiment analysis using **pure PyTorch** (no HuggingFace Trainer).

### Workflow:
1. **Exploratory Data Analysis (EDA)** - Load dataset and visualize class distribution
2. **Model Fine-tuning** - Train BERT with custom PyTorch training loop
3. **Evaluation** - Calculate metrics and confusion matrix
4. **Inference Pipeline** - Create prediction function and test on custom examples

## 1. Setup and Imports

In [ ]:
# Install required packages (uncomment if needed)
# !pip install torch transformers datasets scikit-learn matplotlib seaborn tqdm

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path

# Import custom modules
import sys
sys.path.append('src')

from src.data_loader import (
    load_and_prepare_dataset, 
    create_data_loaders, 
    get_class_distribution
)
from src.model import initialize_model
from src.trainer import train_model, evaluate, save_model
from src.evaluator import (
    calculate_metrics, 
    print_metrics,
    plot_confusion_matrix, 
    plot_class_distribution,
    plot_training_history,
    print_classification_report
)
from src.inference import create_inference_pipeline, test_custom_examples

warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Set device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Exploratory Data Analysis (EDA)

Load the dataset and analyze its characteristics.

In [ ]:
# Load and prepare dataset
print("Loading dataset...")
train_dataset, val_dataset, test_dataset, metadata = load_and_prepare_dataset(
    dataset_name="Sp1786/multiclass-sentiment-analysis-dataset",
    tokenizer_name="bert-base-uncased",
    max_length=128
)

print("\nDataset Information:")
print(f"Number of classes: {metadata['num_classes']}")
print(f"Train samples: {metadata['train_size']}")
print(f"Validation samples: {metadata['val_size']}")
print(f"Test samples: {metadata['test_size']}")

### 2.1 Visualize Class Distribution

In [ ]:
# Get class distribution
class_dist = get_class_distribution(train_dataset)

# Map class indices to names if available
class_names = [f"Class {i}" for i in range(metadata['num_classes'])]

# Plot class distribution
plot_class_distribution(
    class_dist, 
    class_names=class_names,
    save_path='outputs/class_distribution.png'
)

### 2.2 Sample Data Inspection

In [ ]:
# Load original dataset to inspect samples
from datasets import load_dataset

dataset = load_dataset("Sp1786/multiclass-sentiment-analysis-dataset")
text_col = metadata['text_column']
label_col = metadata['label_column']

# Display sample data
print("\nSample Data:")
print("="*80)
for i in range(5):
    sample = dataset['train'][i]
    print(f"\nExample {i+1}:")
    print(f"Text: {sample[text_col][:200]}...")
    print(f"Label: {sample[label_col]} ({class_names[sample[label_col]]})")
    print("-"*80)

## 3. Model Fine-tuning

Fine-tune BERT using **pure PyTorch** (no HuggingFace Trainer).

In [ ]:
# Create data loaders
train_loader, val_loader, test_loader = create_data_loaders(
    train_dataset, val_dataset, test_dataset, batch_size=16
)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

In [ ]:
# Initialize model
model = initialize_model(
    num_classes=metadata['num_classes'],
    model_name='bert-base-uncased',
    dropout=0.3,
    device=device
)

In [ ]:
# Train the model
trained_model, history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=3,
    learning_rate=2e-5,
    device=device
)

### 3.1 Visualize Training History

In [ ]:
# Plot training history
plot_training_history(history, save_path='outputs/training_history.png')

### 3.2 Save Model

In [ ]:
# Save the trained model
Path('models').mkdir(exist_ok=True)
save_model(trained_model, 'models/bert_sentiment_classifier.pt')

## 4. Evaluation

Evaluate the model on the test set and report metrics.

In [ ]:
# Evaluate on test set
import torch.nn as nn

criterion = nn.CrossEntropyLoss()
test_loss, test_acc, y_pred, y_true = evaluate(
    trained_model, test_loader, criterion, device, desc="Testing"
)

print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

### 4.1 Calculate Metrics

In [ ]:
# Calculate and print metrics
metrics = calculate_metrics(y_true, y_pred)
print_metrics(metrics)

### 4.2 Confusion Matrix

In [ ]:
# Plot confusion matrix
plot_confusion_matrix(
    y_true, y_pred, 
    class_names=class_names,
    save_path='outputs/confusion_matrix.png'
)

### 4.3 Detailed Classification Report

In [ ]:
# Print detailed classification report
print_classification_report(y_true, y_pred, class_names=class_names)

## 5. Inference Pipeline

Create a simple `predict_text()` function for inference.

In [ ]:
# Create inference pipeline
inference_pipeline = create_inference_pipeline(
    model=trained_model,
    tokenizer=metadata['tokenizer'],
    class_names=class_names,
    device=device,
    max_length=128
)

# Create the predict_text function
def predict_text(text: str) -> dict:
    """
    Predict sentiment for a given text.
    
    Args:
        text: Input text string
    
    Returns:
        Dictionary with predicted label and confidence score
    """
    result = inference_pipeline.predict_text(text)
    return {
        'predicted_label': result['predicted_label'],
        'confidence': result['confidence']
    }

print("Inference pipeline ready!")

### 5.1 Test with Custom Examples

Test the `predict_text()` function with 5 custom examples.

In [ ]:
# Define custom test examples
custom_examples = [
    "This product exceeded all my expectations! Absolutely amazing quality and fast delivery.",
    "Terrible experience. The item arrived broken and customer service was unhelpful.",
    "It's okay, nothing special. Does what it's supposed to do but nothing more.",
    "I'm extremely disappointed with this purchase. Complete waste of money.",
    "Great value for money! Would definitely recommend to friends and family."
]

# Test with custom examples
test_custom_examples(inference_pipeline, custom_examples)

### 5.2 Quick Test of predict_text() Function

In [ ]:
# Quick test
test_text = "This is absolutely wonderful! Best purchase ever!"
result = predict_text(test_text)

print(f"Text: {test_text}")
print(f"Predicted Label: {result['predicted_label']}")
print(f"Confidence: {result['confidence']:.4f}")

## 6. Summary

### Key Achievements:
✅ Loaded and analyzed the multiclass sentiment analysis dataset  
✅ Visualized class distribution and identified any imbalance  
✅ Fine-tuned BERT using **pure PyTorch** (no HuggingFace Trainer)  
✅ Evaluated model with Accuracy, Precision, Recall, and F1-Score  
✅ Generated confusion matrix visualization  
✅ Created `predict_text()` inference function  
✅ Tested on 5 custom examples  

### Implementation Details:
- **Framework:** Pure PyTorch with custom training loop
- **Model:** BERT-base-uncased
- **Optimizer:** AdamW with linear warmup and decay
- **Training:** 3 epochs with early stopping based on validation accuracy

### Files Generated:
- `outputs/class_distribution.png` - Class distribution visualization
- `outputs/confusion_matrix.png` - Confusion matrix
- `outputs/training_history.png` - Training curves
- `models/bert_sentiment_classifier.pt` - Trained model weights

---

## Submission Information

**Team Name:** Lost in Translation  
**Dataset:** multiclass-sentiment-analysis-dataset  
**Repository:** LostInTranslation_multiclass-sentiment-analysis-dataset  
**Deadline:** Monday, Feb 16th, 10 AM  

All code is available in the GitHub repository with clear documentation in README.md.